In [0]:
%pip install azure-eventhub

  Obtaining dependency information for azure-eventhub from https://files.pythonhosted.org/packages/30/ba/b54e0f76384b00a8682b1e041673f43947f668dbc8508f026d67446a7d2f/azure_eventhub-5.15.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/317.1 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 13.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# ==========================================
# Event Hub Connection
# ==========================================

connectionString = (
    "Endpoint=sb://socialmedia-eventhub.servicebus.windows.net/;"
    "SharedAccessKeyName=RootManageSharedAccessKey;"
    "SharedAccessKey=nCxt6nE1qV3pyZHbmFFh7BZBScVdLQSaP+AEhOq4lB0=;"
    "EntityPath=tweets-hub"
)

ehConf = {}

ehConf["eventhubs.connectionString"] = (
    sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString)
)

ehConf["eventhubs.consumerGroup"] = "$Default"

startingEventPosition = {
    "offset": "-1",
    "seqNo": -1,
    "enqueuedTime": None,
    "isInclusive": True
}

ehConf["eventhubs.startingPosition"] = json.dumps(startingEventPosition)
ehConf["eventhubs.maxEventsPerTrigger"] = "5000"

# ==========================================
# Read Stream
# ==========================================

rawDF = (
    spark.readStream
         .format("eventhubs")
         .options(**ehConf)
         .load()
)

eventDF = rawDF.selectExpr(
    "CAST(body AS STRING) AS message"
)

# ==========================================
# Schema
# ==========================================

schema = StructType([
    StructField("tweet_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("tweet_text", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("timestamp_1", StringType(), True),
    StructField("likes", DoubleType(), True),
    StructField("retweets", DoubleType(), True),
    StructField("replies", DoubleType(), True),
    StructField("impressions", DoubleType(), True),
    StructField("engagement", DoubleType(), True)
])

# ==========================================
# Parse JSON
# ==========================================

parsedDF = (
    eventDF
        .select(from_json(col("message"), schema).alias("data"))
        .select("data.*")
)

# ==========================================
# Error Records
# ==========================================

errorDF = parsedDF.filter(col("tweet_id").isNull())

errorQuery = (
    errorDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://socialmedia@socialmediaadls001.dfs.core.windows.net/checkpoints/error_bronze_tweets"
        )
        .toTable("bronze_catalog1.error.error_bronze_tweets")
)

# ==========================================
# Bronze Data
# ==========================================

bronzeDF = (
    parsedDF
    .filter(col("tweet_id").isNotNull())

    .withColumn(
        "timestamp_1",
        to_timestamp(col("timestamp_1"), "dd-MM-yyyy HH:mm")
    )

    .withColumn("bronze_load_time", current_timestamp())
    .withColumn("pipeline_name", lit("Bronze_Tweets"))
    .withColumn("source_system", lit("Azure Event Hub"))
    .withColumn("ingestion_date", current_date())

    .withWatermark("timestamp_1", "10 minutes")
)

# ==========================================
# Write Bronze Table
# ==========================================

bronzeQuery = (
    bronzeDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://socialmedia@socialmediaadls001.dfs.core.windows.net/checkpoints/bronze_tweets1"
        )
        .option("mergeSchema", "true")
        .toTable("bronze_catalog1.raw.bronze_tweets_raw")
)

bronzeQuery.awaitTermination()
errorQuery.awaitTermination()

print("Bronze Tweets Loaded Successfully")

Bronze Tweets Loaded Successfully


In [0]:
%sql
SELECT COUNT(*) FROM bronze_catalog1.raw.bronze_tweets_raw;

count(1)
5000


In [0]:
%sql
SELECT COUNT(*) FROM bronze_catalog1.error.error_bronze_tweets;

count(1)
0


In [0]:
%sql
SELECT * 
FROM bronze_catalog1.raw.bronze_tweets_raw;

tweet_id,user_id,tweet_text,timestamp,timestamp_1,likes,retweets,replies,impressions,engagement,bronze_load_time,pipeline_name,source_system,ingestion_date
T24019,8066.0,Love this!!! 😊,api,2025-01-02T02:19:00Z,18022.0,173.0,760.0,1677.0,1374.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T32368,6805.0,Good & Bad mixed!!!,android,2025-01-03T07:15:00Z,11258.0,2134.0,380.0,1759.0,NaN,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T13032,2244.0,Good & Bad mixed!!!,api,2025-01-02T01:30:00Z,18422.0,1267.0,990.0,2367.0,NaN,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T13908,5031.0,Love this!!! 😊,ios,2025-01-22T20:46:00Z,10701.0,4507.0,153.0,606.0,1218.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T22901,6393.0,Good & Bad mixed!!!,android,2025-01-08T07:53:00Z,2372.0,3128.0,939.0,2301.0,644.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T37067,6128.0,Error###Detected,web,2025-01-12T01:25:00Z,NaN,1602.0,251.0,2460.0,1808.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T37306,9926.0,Testing!!!@@@###,android,2025-01-23T03:22:00Z,10443.0,4993.0,395.0,926.0,486.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T29585,9846.0,Love this!!! 😊,ios,2025-01-23T20:53:00Z,12627.0,892.0,923.0,2808.0,1357.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T7249,9046.0,Great product!!! #AI,api,2025-01-17T14:08:00Z,10976.0,NaN,232.0,2369.0,1856.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T37623,8632.0,Worst service ever :(,android,2025-01-06T09:58:00Z,5435.0,4825.0,NaN,1178.0,1540.0,2026-07-09T09:52:24.541Z,Bronze_Tweets,Azure Event Hub,2026-07-09


In [0]:
%sql
DROP TABLE IF EXISTS bronze_catalog1.raw.bronze_tweets_raw;
DROP TABLE IF EXISTS bronze_catalog1.error.error_bronze_tweets;

In [0]:
%sql
DESCRIBE bronze_catalog1.raw.bronze_tweets_raw;

col_name,data_type,comment
tweet_id,string,null
user_id,string,null
tweet_text,string,null
timestamp,string,null
timestamp_1,timestamp,null
likes,double,null
retweets,double,null
replies,double,null
impressions,double,null
engagement,double,null


In [0]:
%sql
SHOW CATALOGS;

catalog
bronze_catalog1
gold_catalog1
samples
silver_catalog1
social_media_databricks
system


In [0]:
%sql
SHOW SCHEMAS IN bronze_catalog1;

databaseName
default
error
information_schema
raw


In [0]:
%sql
SHOW TABLES IN bronze_catalog1.raw;

database,tableName,isTemporary
raw,bronze_sentiment1,false
raw,bronze_trends,false
raw,bronze_tweets_raw,false
raw,bronze_user_metadata1,false
raw,bronze_user_metadata_raw,false
raw,bronze_valid_tweets1,false
,_sqldf,true
